# Your First Reflexio Workflow

> **Time:** ~5 minutes | **Level:** Beginner

In this notebook you'll:
1. **Publish** a customer support conversation to Reflexio
2. **Retrieve** auto-extracted user profiles
3. **Build** a memory-enhanced prompt for your LLM

### Prerequisites
- Reflexio server running (`uv run reflexio services start --only backend`)
- `OPENAI_API_KEY` set in your `.env` file

> **Storage:** SQLite is used by default — no database setup needed.

> **Install:** If running outside the project, install first: `pip install reflexio-client rich pandas`

In [1]:
import uuid

from _display_helpers import *

from reflexio import InteractionData, ReflexioClient, UserActionType

# Each run uses a unique ID so the notebook is idempotent
RUN_ID = uuid.uuid4().hex[:8]
USER_ID = f"demo_customer_{RUN_ID}"

url, api_key = load_env()
client = ReflexioClient(url_endpoint=url, api_key=api_key)

OK Environment loaded — server URL: http://localhost:8081

## Publish a Conversation

**Interactions** are how Reflexio learns about users. Let's publish a realistic customer support conversation — the customer returns a laptop, mentions preferences, and asks about an upgrade.

In [2]:
response = client.publish_interaction(
    user_id=USER_ID,
    interactions=[
        InteractionData(
            role="User",
            content="Hi, I bought an Acme ProBook laptop last week but the screen has a dead pixel. Can I return it?",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="I'm sorry to hear about the dead pixel! Yes, you can return it within 30 days of purchase. Would you like me to start a return for you?",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="User",
            content="Yes please. Also, I'd prefer to get updates by email rather than phone — I'm usually in meetings during the day.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="Got it! I'll set email as your preferred contact method. Before I start the return, could you confirm your order number?",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="User",
            content="It's ORD-8834. And while we're at it, I'm actually interested in upgrading to the ProBook Max. My budget is around $2,500.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="Great choice! The ProBook Max starts at $2,299 and fits your budget perfectly. I'll include upgrade options in your return confirmation email.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="User",
            content="Perfect. One more thing — I need something with a quiet keyboard. I do a lot of video calls and my colleagues complain about typing noise.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="The ProBook Max has a silent membrane keyboard designed for open-office and video call environments. I'll note that preference on your account. You'll receive the return label and upgrade details by email within 24 hours.",
            user_action=UserActionType.NONE,
        ),
    ],
    source="notebook",
    session_id=f"session_{RUN_ID}",
    wait_for_response=True,
)
show_success(f"Published 8 interaction turns for user '{USER_ID}'")

OK Published 8 interaction turns for user 'demo_customer_7acd8394'

## See What Reflexio Learned

Reflexio automatically extracts **user profiles** — facts and preferences about each user — from conversations. Let's see what it found.

In [3]:
profile_resp = client.get_profiles(force_refresh=True, user_id=USER_ID)
show_profiles(profile_resp.user_profiles)

### User Profiles

,#,Content,Extractor,Status
0,1,Owns an Acme ProBook laptop purchased around O...,default_profile_extractor,current
1,2,Prefers communication via email rather than ph...,default_profile_extractor,current
2,3,Order number for current return is ORD-8834,default_profile_extractor,current
3,4,Looking to upgrade to a ProBook Max with a bud...,default_profile_extractor,current
4,5,Requires a quiet keyboard due to frequent vide...,default_profile_extractor,current


## Search Profiles with Natural Language

Instead of listing all profiles, you can use **semantic search** to find specific information. Reflexio matches meaning, not just keywords.

In [ ]:
search_resp = client.search_profiles(
    user_id=USER_ID,
    query="communication preferences",
    top_k=5,
)
show_profiles(search_resp.user_profiles, title="Search Results: 'communication preferences'")

## Build a Memory-Enhanced Prompt

> This is the core pattern: **retrieve** user context from Reflexio, then **inject** it into your LLM prompt.

This is how your agent gets smarter over time — each conversation adds to the user's profile, and future responses are personalized.

In [ ]:
# Retrieve profiles and agent playbooks
profiles = client.search_profiles(user_id=USER_ID, query="preferences", top_k=5).user_profiles
playbooks_resp = client.get_agent_playbooks(limit=5)
playbooks = playbooks_resp.agent_playbooks if playbooks_resp.success else []

# Build the enhanced prompt
profile_context = "".join(f"- {p.profile_content}" for p in profiles)
playbook_context = "".join(f"- {pb.content}" for pb in playbooks)

prompt = f"""You are an Acme Electronics support agent.

## What you know about this user:
{profile_context or "No profile information yet."}

## Guidelines from past interactions:
{playbook_context or "No playbook entries yet."}

## User message:
What accessories would you recommend for my laptop?
"""

display(Markdown("### Generated Prompt"))
print(prompt)

## Cleanup (Optional)

Remove the demo data created by this notebook.

In [ ]:
try:
    client.delete_all_interactions()
    client.delete_all_profiles()
    client.delete_all_playbooks()
    show_success("Demo data cleaned up")
except Exception:
    pass  # Safe to skip

## Summary & Next Steps

You just completed the core Reflexio workflow:
1. Published interactions -- Reflexio auto-extracted user profiles
2. Searched profiles using natural language
3. Built an LLM prompt enhanced with user memory

### Continue Learning

| Notebook | What you'll learn |
|----------|-------------------|
| [01 -- Interactions](01_interactions.ipynb) | All interaction types: text, tool use, user actions |
| [02 -- Profiles](02_profiles.ipynb) | Profile lifecycle, multi-user search, change logs |
| [03 -- Playbooks](03_playbook.ipynb) | Agent playbook extraction and aggregation |
| [04 -- Configuration](04_configuration.ipynb) | Custom extractors, tools, and evaluation |
| [07 -- LangChain](07_langchain_integration.ipynb) | Integrate Reflexio with LangChain agents |
| [05 -- Concurrent Sessions](05_concurrent_sessions.ipynb) | Multi-user load and data isolation |
| [06 -- Simulation](06_real_world_simulation.ipynb) | Watch Reflexio learn from real conversations |